In [31]:
import os
from dotenv import load_dotenv
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline

# Load environment variables
load_dotenv()


True

In [32]:


# Get API keys securely
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found. Check your .env file.")

if not HF_TOKEN:
    raise ValueError("HUGGINGFACE_API_KEY not found. Check your .env file.")


In [33]:
def load_pdf_file(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents

extracted_data = load_pdf_file("C:/Bits_hyd/doctor_pov/CuraMateDR/Data/tb.pdf")
print("Number of Pages Loaded:", len(extracted_data))


Number of Pages Loaded: 1874


In [5]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks = text_split(extracted_data)
print("Number of Text Chunks:", len(text_chunks))


Number of Text Chunks: 18109


In [6]:
def get_hugging_face_embedding():
    return HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

embeddings = get_hugging_face_embedding()
print("Embeddings Loaded Successfully")


Embeddings Loaded Successfully


In [10]:
print("Available Indexes:", pc.list_indexes())


Available Indexes: {'indexes': [{'deletion_protection': 'disabled',
              'dimension': 384,
              'host': 'medibot-vit7otl.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'medibot',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}}]}


In [11]:
from pinecone import Pinecone

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY, environment="us-east-1")

# Connect to the existing index
index_name = "medibot"
index = pc.Index(index_name)

# Check index stats
print("Index is ready. Details:", index.describe_index_stats())


Index is ready. Details: {'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 36218}},
 'total_vector_count': 36218}


In [14]:
from langchain_pinecone import PineconeVectorStore

# Connect to the existing index
docsearch = PineconeVectorStore.from_existing_index(
    index_name="medibot", 
    embedding=embeddings
)

# Create retriever
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 4})
print("Retriever Created Successfully")


Retriever Created Successfully


In [ ]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_name, 
                                             token=HF_TOKEN,
                                             torch_dtype=torch.float16, 
                                             device_map="auto")

print("LLaMA Model Loaded Successfully")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


LLaMA Model Loaded Successfully


In [34]:
# Wrap model in a Hugging Face text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=500  # Limit response length
)

# Create a LangChain LLM wrapper
llm = HuggingFacePipeline(pipeline=pipe)
print("✅ LLaMA Wrapped in LangChain")


Device set to use cpu


✅ LLaMA Wrapped in LangChain


C:\Users\Neranjana\AppData\Local\Temp\ipykernel_35300\2217331991.py:10: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [36]:

# System Prompt
system_prompt = (
    "You are an AI assistant for medical question answering."
    "Use the retrieved context to answer accurately."
    "If unsure, say 'I don't know'."
    "Keep responses concise (max 4 sentences)."
    "\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)




In [37]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ RAG Pipeline Created Successfully")


✅ RAG Pipeline Created Successfully


In [ ]:
input_data = {"input": "What is acne?"}  # Correct input format

response = rag_chain.invoke(input_data)  # Correct method invocation
print("🩺 Answer:", response.get("answer", "No answer generated."))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
